In [1]:
# Make repaired datasets for all california jurisdictions
# SLOW!

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts"))
from data_repair import data_repair, _slugify

SUMMARY_FILEPATH = os.path.join(MY_DATA_PATH, f"dewey_summary.parquet")

COLUMNS = [
    'PERMIT_NUMBER', 'JURISDICTION', 'STATE', 
    'FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE', 
    'STATUS_NORMALIZED', 'STATUS_ORIGINAL', 
    'RECORD_TYPE_ORIGINAL', 'RECORD_SUBTYPE_ORIGINAL', 
    'APN', 'STREET', 'ZIPCODE', 'COUNTY_FIPS', 'CBSA_FIPS',
    'DESCRIPTION', 'DATA'
]  # Columns to load from data file

OUTPUT_DIR = os.path.join(MY_DATA_PATH, "processed_data")
REPLACE = True   # Whether to replace existing output files


In [2]:
# Load the summary file
summ_df = pd.read_parquet(SUMMARY_FILEPATH)
summ_df = summ_df.loc[summ_df["STATE"] == "CA"]

In [3]:
# Jurisdiction / state
j_df = summ_df[['JURISDICTION', 'STATE']].drop_duplicates()
jurisdictions = j_df['JURISDICTION'].tolist()
states = j_df['STATE'].tolist()

In [4]:
# Iterate through jurisdictions

t0 = time.time()
for i, (jurisdiction, state) in enumerate(zip(jurisdictions, states)):
    jurisdiction_slug = _slugify(jurisdiction)
    state_slug = state.lower().strip()
    os.makedirs(os.path.join(OUTPUT_DIR, f"{state_slug}"), exist_ok=True)
    output_filepath = os.path.join(OUTPUT_DIR, f"{state_slug}", f"{state_slug}_{jurisdiction_slug}_repaired.parquet")

    df = du.get_data_for_jurisdiction(
        jurisdiction=jurisdiction,
        state=state,
        columns=COLUMNS,
        repair=True,
        deduplicate=True,
        save_to=output_filepath,
        replace=REPLACE,
        remove_raw_data_col=True
    )

    dt = (time.time() - t0) / 60
    print(f"Total elapsed time: {dt:.2f} minutes")


Retrieving data for Alameda CA ... 11/11 files ... elapsed time 62.00 seconds              
0 duplicates dropped from 157,093 original records
Data saved to /Users/ekung/Dropbox/projects/la-permits-data/processed_data/ca/ca_alameda_repaired.parquet
Total elapsed time: 1.22 minutes
Retrieving data for Alameda County CA ... 7/7 files ... elapsed time 56.36 seconds              
2 duplicates dropped from 151,441 original records
Data saved to /Users/ekung/Dropbox/projects/la-permits-data/processed_data/ca/ca_alameda_county_repaired.parquet
Total elapsed time: 2.23 minutes
Retrieving data for Albany CA ... 1/1 files ... elapsed time 0.00 seconds              
0 duplicates dropped from 4,408 original records
Data saved to /Users/ekung/Dropbox/projects/la-permits-data/processed_data/ca/ca_albany_repaired.parquet
Total elapsed time: 2.24 minutes
Retrieving data for Alhambra CA ... 2/2 files ... elapsed time 2.16 seconds              
38 duplicates dropped from 16,110 original records
Data sav